### Import statements

In [ ]:
from speech_signal import run_training, plot_training, rolling_samples, plot_rolling, feature_count, all_prefs
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib import pyplot as plt

### Initialize state

In [ ]:
state = {}

### Define window view controls

In [ ]:
# work dropdown
work_dd = widgets.Dropdown(
    description = "Work",
    options = [work for work in all_prefs],
)

# prefix dropdown
pref_dd = widgets.Dropdown(
    description = "Book",
    options = [pref for pref in all_prefs[work_dd.value]],
)

# figure viewport
roll_view = widgets.Output()

# handler for work change
def on_work_change(change):
    '''on change to work, update pref options'''
    pref_dd.options = [pref for pref in all_prefs[work_dd.value]]

    # trigger pref update
    on_pref_change(None)

# handler for any change to view
def on_pref_change(change):
    '''redraw the rolling window plot'''
    
    if "test" not in state:
        return
    with roll_view:
        clear_output(wait=True)
        fig = plot_rolling(state["test"], work=work_dd.value, pref=pref_dd.value)
        display(fig)
        plt.close(fig)

# register handlers with widgets
work_dd.observe(on_work_change, names="value")
pref_dd.observe(on_pref_change, names="value")

### Define feature selection controls

In [ ]:
# dictionary holds per-feature-class feature input
feat_input = {}

# define the inputs (Textarea for now)
for label in feature_count:
    feat_input[label] = widgets.Textarea(
        value = "\n".join([feat.strip() for feat in feature_count[label].index.values]),
        description = "Include:",
    )

# define the tabbed container to hold them
tab_container = widgets.Tab()
tab_container.children = [ta for ta in feat_input.values()]
tab_container.titles = [label for label in feat_input]

# a function to validate contents and return a feature set dict
def get_valid_features():
    '''Parse the contents of textareas and extract valid features by feature class'''

    fs = {}

    for label in feat_input:
        fs[label] = [feat for feat in feat_input[label].value.split() if feat in feature_count[label].index.values]

    return fs

### Define training controls

In [ ]:
# sample size
sample_size_input = widgets.BoundedIntText(
    value = 1000,
    min = 200,
    max = 5000,
    description = "Sample size"
)

# random number seed
seed_input = widgets.BoundedIntText(
    value = 0,
    min = 0,
    max = 2**53 - 1,
    description = "Seed"
)

# run button
train_btn = widgets.Button(
    description = "Run training",
)

# model viewport
train_view = widgets.Output()

# handler for button click
def on_train_click(btn):
    '''train model, project rolling samples'''

    # for now just use base features
    feature_set = get_valid_features()

    # run training
    state["train"] = run_training(
        feature_set = feature_set,
        sample_size = sample_size_input.value,
        seed = seed_input.value,
    )

    # display model
    with train_view:
        clear_output(wait=True)
        fig = plot_training(state["train"])
        display(fig)
        plt.close(fig)

    # calculate rolling window and project
    state["test"] = rolling_samples(state["train"])

    # trigger rolling viewport update
    on_pref_change(None)

# register handler for train button
train_btn.on_click(on_train_click)

## Interactive UI

### Feature Selection

In [ ]:
display(tab_container)

### Training

In [ ]:
display(widgets.HBox([sample_size_input, seed_input, train_btn]))
display(train_view)

### Rolling Samples

In [ ]:
display(widgets.HBox([work_dd, pref_dd]))
display(roll_view)